In [3]:
# =====================================================
# IMPORTS
# =====================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

# =====================================================
# MOUNT DRIVE
# =====================================================

drive.mount('/content/drive')

# =====================================================
# LOAD DATA
# =====================================================

DATA_PATH = "/content/drive/MyDrive/Dataset/delivery_data.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)

df.head()

Mounted at /content/drive
Shape: (144867, 24)


,data,trip_creation_time,route_schedule_uuid,route_type,trip_uuid,source_center,source_name,destination_center,destination_name,od_start_time,...,cutoff_timestamp,actual_distance_to_destination,actual_time,osrm_time,osrm_distance,factor,segment_actual_time,segment_osrm_time,segment_osrm_distance,segment_factor
0,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 04:27:55,10.435660,14.0,11.0,11.9653,1.272727,14.0,11.0,11.9653,1.272727
1,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 04:17:55,18.936842,24.0,20.0,21.7243,1.200000,10.0,9.0,9.7590,1.111111
2,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 04:01:19.505586,27.637279,40.0,28.0,32.5395,1.428571,16.0,7.0,10.8152,2.285714
3,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 03:39:57,36.118028,62.0,40.0,45.5620,1.550000,21.0,12.0,13.0224,1.750000
4,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 03:33:55,39.386040,68.0,44.0,54.2181,1.545455,6.0,5.0,3.9153,1.200000


In [4]:
print("="*50)
print("DATASET OVERVIEW")
print("="*50)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nRoute Types")
print(df["route_type"].value_counts())

print("\nUnique Trips")
print(df["trip_uuid"].nunique())

print("\nUnique Facilities")

facilities = set(df["source_center"]) | set(df["destination_center"])

print(len(facilities))

DATASET OVERVIEW
Rows: 144867
Columns: 24

Route Types
route_type
FTL        99660
Carting    45207
Name: count, dtype: int64

Unique Trips
14817

Unique Facilities
1657


In [5]:
# =====================================================
# TARGET VALIDATION
# =====================================================

df["factor_check"] = (
    df["actual_time"] /
    df["osrm_time"]
)

error = (
    df["factor"] -
    df["factor_check"]
).abs()

print(
    "Validation Accuracy:",
    (error < 0.001).mean()
)

df["factor"].describe()

Validation Accuracy: 1.0


,factor
count,144867.000000
mean,2.120107
std,1.715421
min,0.144000
25%,1.604264
50%,1.857143
75%,2.213483
max,77.387097


In [6]:
# =====================================================
# LEAKAGE AUDIT
# =====================================================

leakage_columns = [
    "actual_time",
    "segment_actual_time",
    "od_end_time"
]

print("Leakage Columns")

for col in leakage_columns:
    print(col)

Leakage Columns
actual_time
segment_actual_time
od_end_time


In [7]:
datetime_cols = [
    "trip_creation_time",
    "od_start_time",
    "od_end_time",
    "cutoff_timestamp"
]

for col in datetime_cols:
    print("\n", col)
    print(df[col].head())

df["trip_creation_time"].head(10)



for col in datetime_cols:
    df[col] = pd.to_datetime(
        df[col],
        format="mixed",
        errors="coerce"
    )

df[datetime_cols].isna().sum()


 trip_creation_time
0    2018-09-20 02:35:36.476840
1    2018-09-20 02:35:36.476840
2    2018-09-20 02:35:36.476840
3    2018-09-20 02:35:36.476840
4    2018-09-20 02:35:36.476840
Name: trip_creation_time, dtype: object

 od_start_time
0    2018-09-20 03:21:32.418600
1    2018-09-20 03:21:32.418600
2    2018-09-20 03:21:32.418600
3    2018-09-20 03:21:32.418600
4    2018-09-20 03:21:32.418600
Name: od_start_time, dtype: object

 od_end_time
0    2018-09-20 04:47:45.236797
1    2018-09-20 04:47:45.236797
2    2018-09-20 04:47:45.236797
3    2018-09-20 04:47:45.236797
4    2018-09-20 04:47:45.236797
Name: od_end_time, dtype: object

 cutoff_timestamp
0           2018-09-20 04:27:55
1           2018-09-20 04:17:55
2    2018-09-20 04:01:19.505586
3           2018-09-20 03:39:57
4           2018-09-20 03:33:55
Name: cutoff_timestamp, dtype: object


,0
trip_creation_time,0
od_start_time,0
od_end_time,0
cutoff_timestamp,0


In [8]:
df["cutoff_timestamp"] = pd.to_datetime(
    df["cutoff_timestamp"],
    format="mixed"
)

df[datetime_cols].dtypes

,0
trip_creation_time,datetime64[ns]
od_start_time,datetime64[ns]
od_end_time,datetime64[ns]
cutoff_timestamp,datetime64[ns]


In [9]:
df.info()
df.columns.tolist()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144867 entries, 0 to 144866
Data columns (total 25 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   data                            144867 non-null  object        
 1   trip_creation_time              144867 non-null  datetime64[ns]
 2   route_schedule_uuid             144867 non-null  object        
 3   route_type                      144867 non-null  object        
 4   trip_uuid                       144867 non-null  object        
 5   source_center                   144867 non-null  object        
 6   source_name                     144574 non-null  object        
 7   destination_center              144867 non-null  object        
 8   destination_name                144606 non-null  object        
 9   od_start_time                   144867 non-null  datetime64[ns]
 10  od_end_time                     144867 non-null  datetim

['data',
 'trip_creation_time',
 'route_schedule_uuid',
 'route_type',
 'trip_uuid',
 'source_center',
 'source_name',
 'destination_center',
 'destination_name',
 'od_start_time',
 'od_end_time',
 'start_scan_to_end_scan',
 'is_cutoff',
 'cutoff_factor',
 'cutoff_timestamp',
 'actual_distance_to_destination',
 'actual_time',
 'osrm_time',
 'osrm_distance',
 'factor',
 'segment_actual_time',
 'segment_osrm_time',
 'segment_osrm_distance',
 'segment_factor',
 'factor_check']

In [10]:
# =====================================================
# CREATE WORKING COPY
# =====================================================

feature_df = df.copy()

print("Original:", df.shape)
print("Working :", feature_df.shape)

Original: (144867, 25)
Working : (144867, 25)


In [11]:
# =====================================================
# DATETIME FEATURES
# =====================================================

feature_df["trip_hour"] = (
    feature_df["trip_creation_time"]
    .dt.hour
)

feature_df["trip_weekday"] = (
    feature_df["trip_creation_time"]
    .dt.weekday
)

feature_df["trip_month"] = (
    feature_df["trip_creation_time"]
    .dt.month
)

feature_df["is_weekend"] = (
    feature_df["trip_weekday"] >= 5
).astype(int)

# =====================================================
# OPERATIONAL TIMING FEATURES
# =====================================================

feature_df["trip_to_scan_minutes"] = (
    (
        feature_df["od_start_time"]
        -
        feature_df["trip_creation_time"]
    )
    .dt.total_seconds()
    / 60
)

feature_df["trip_to_cutoff_minutes"] = (
    (
        feature_df["cutoff_timestamp"]
        -
        feature_df["trip_creation_time"]
    )
    .dt.total_seconds()
    / 60
)

# =====================================================
# OSRM FEATURES
# =====================================================

feature_df["osrm_speed"] = (
    feature_df["osrm_distance"]
    /
    (feature_df["osrm_time"] + 1)
)

# =====================================================
# FACILITY VOLUME FEATURES
# =====================================================

source_volume = (
    feature_df.groupby("source_center")
    .size()
)

destination_volume = (
    feature_df.groupby("destination_center")
    .size()
)

feature_df["source_volume"] = (
    feature_df["source_center"]
    .map(source_volume)
)

feature_df["destination_volume"] = (
    feature_df["destination_center"]
    .map(destination_volume)
)

# =====================================================
# CORRIDOR FEATURES
# =====================================================

feature_df["corridor"] = (
    feature_df["source_center"]
    + "_TO_" +
    feature_df["destination_center"]
)

corridor_volume = (
    feature_df.groupby("corridor")
    .size()
)

feature_df["corridor_volume"] = (
    feature_df["corridor"]
    .map(corridor_volume)
)

print(feature_df.shape)

(144867, 36)


In [12]:
feature_cols = [
    "trip_hour",
    "trip_weekday",
    "trip_month",
    "is_weekend",
    "trip_to_scan_minutes",
    "trip_to_cutoff_minutes",
    "osrm_speed",
    "source_volume",
    "destination_volume",
    "corridor_volume"
]
feature_df[feature_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
trip_hour,144867.0,12.652516,7.949513,0.000000,5.000000,15.000000,20.000000,23.000000
trip_weekday,144867.0,2.918097,1.936452,0.000000,1.000000,3.000000,5.000000,6.000000
trip_month,144867.0,9.120925,0.326041,9.000000,9.000000,9.000000,9.000000,10.000000
is_weekend,144867.0,0.260970,0.439165,0.000000,0.000000,0.000000,1.000000,1.000000
trip_to_scan_minutes,144867.0,268.369924,481.058549,0.000000,0.000000,0.000000,375.823250,3575.014669
trip_to_cutoff_minutes,144867.0,778.095570,814.536221,0.000000,187.203151,469.872756,1120.404752,7356.031174
osrm_speed,144867.0,1.215448,0.188401,0.400648,1.071712,1.294960,1.366780,1.563192
source_volume,144867.0,5620.646973,8347.220322,1.000000,90.000000,1331.000000,9088.000000,23347.000000
destination_volume,144867.0,3628.878206,4972.185743,1.000000,99.000000,1232.000000,5142.000000,15192.000000
corridor_volume,144867.0,592.017775,1066.126232,1.000000,51.000000,154.000000,665.000000,4976.000000


In [13]:
feature_df.to_csv(
    "/content/feature_store.csv",
    index=False
)

In [14]:
feature_df[
    [
        "trip_to_scan_minutes",
        "trip_to_cutoff_minutes",
        "source_volume",
        "destination_volume",
        "corridor_volume"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
trip_to_scan_minutes,144867.0,268.369924,481.058549,0.0,0.000000,0.000000,375.823250,3575.014669
trip_to_cutoff_minutes,144867.0,778.095570,814.536221,0.0,187.203151,469.872756,1120.404752,7356.031174
source_volume,144867.0,5620.646973,8347.220322,1.0,90.000000,1331.000000,9088.000000,23347.000000
destination_volume,144867.0,3628.878206,4972.185743,1.0,99.000000,1232.000000,5142.000000,15192.000000
corridor_volume,144867.0,592.017775,1066.126232,1.0,51.000000,154.000000,665.000000,4976.000000


In [15]:
(
    feature_df["trip_to_scan_minutes"] == 0
).mean()

np.float64(0.5656636777181829)

In [16]:
corr_features = [
    "trip_hour",
    "trip_weekday",
    "trip_month",
    "is_weekend",
    "trip_to_scan_minutes",
    "trip_to_cutoff_minutes",
    "osrm_speed",
    "source_volume",
    "destination_volume",
    "corridor_volume",
    "factor"
]

corr_matrix = (
    feature_df[corr_features]
    .corr(numeric_only=True)
)

corr_matrix["factor"].sort_values(
    ascending=False
)

,factor
factor,1.000000
osrm_speed,0.050542
trip_month,-0.002089
trip_weekday,-0.003636
is_weekend,-0.004760
trip_to_scan_minutes,-0.009115
trip_hour,-0.054341
trip_to_cutoff_minutes,-0.065408
corridor_volume,-0.075245
destination_volume,-0.077740


In [17]:
drop_cols = [
    "factor_check"
]

feature_df = feature_df.drop(
    columns=drop_cols,
    errors="ignore"
)

print(feature_df.shape)

(144867, 35)


In [18]:
feature_df.to_csv(
    "/content/feature_store.csv",
    index=False
)

print("Feature store saved.")

Feature store saved.


In [19]:
drop_cols = [
    "factor_check"
]

feature_df = feature_df.drop(
    columns=drop_cols,
    errors="ignore"
)

print(feature_df.shape)

(144867, 35)


Train/Validation Strategy

Target: factor

Split Method:
- Group split by trip_uuid

Reason:
- Multiple rows belong to the same trip.
- Random row split would leak trip information.

In [21]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        feature_df,
        groups=feature_df["trip_uuid"]
    )
)

train_df = feature_df.iloc[train_idx]
test_df = feature_df.iloc[test_idx]

print(train_df.shape)
print(test_df.shape)

(116451, 35)
(28416, 35)


In [22]:
graph_df = feature_df[
    [
        "source_center",
        "destination_center",
        "factor",
        "route_type",
        "osrm_time",
        "osrm_distance"
    ]
].copy()

In [23]:
graph_df.to_csv(
    "/content/graph_ready_dataset.csv",
    index=False
)

In [24]:
engineered_features = [
    "trip_hour",
    "trip_weekday",
    "trip_month",
    "is_weekend",
    "trip_to_scan_minutes",
    "trip_to_cutoff_minutes",
    "osrm_speed",
    "source_volume",
    "destination_volume",
    "corridor_volume"
]

print("Engineered Features:", len(engineered_features))
print(engineered_features)

Engineered Features: 10
['trip_hour', 'trip_weekday', 'trip_month', 'is_weekend', 'trip_to_scan_minutes', 'trip_to_cutoff_minutes', 'osrm_speed', 'source_volume', 'destination_volume', 'corridor_volume']
